Setup

In [1]:
# Install FFmpeg (for Task 3 MP3 encoding) and optional mido package
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install mido

# Create and enter the lab working directory
import os
os.makedirs("CCS3208_Week3_Lab", exist_ok=True)
%cd CCS3208_Week3_Lab

# Verify Python version and required standard-library modules
import sys, math, wave, struct, zlib, hashlib
from pathlib import Path
from IPython.display import Audio, display

print("Python version:", sys.version)
print("Working directory:", os.getcwd())
!ffmpeg -version | head -n 1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
/content/CCS3208_Week3_Lab
Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Working directory: /content/CCS3208_Week3_Lab
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


Cell 1 — Task 1B: Baseline MIDI-Controlled Audio Synthesis
Generates midi_demo.wav from the three MIDI note events and prints sample count, duration, byte size, and calculated frequencies.


In [2]:
import math, struct, wave
from pathlib import Path
from IPython.display import Audio, display

SAMPLE_RATE = 44100
EVENTS = [
    (0.00, 0.50, 60, 90),
    (0.50, 0.50, 64, 105),
    (1.00, 0.75, 67, 80),
]

def midi_to_freq(note):
    return 440.0 * 2 ** ((note - 69) / 12)

def make_audio(events, filename):
    length = max(start + duration for start, duration, _, _ in events)
    samples = [0.0] * int(length * SAMPLE_RATE)
    for start, duration, note, velocity in events:
        f = midi_to_freq(note)
        begin = int(start * SAMPLE_RATE)
        count = int(duration * SAMPLE_RATE)
        for j in range(count):
            # Short fade-in and fade-out to reduce clicks.
            attack = min(1.0, j / max(1, int(0.01 * SAMPLE_RATE)))
            release = min(1.0, (count - j - 1) / max(1, int(0.02 * SAMPLE_RATE)))
            envelope = max(0.0, min(attack, release))
            value = 0.65 * (velocity / 127) * envelope * math.sin(2 * math.pi * f * j / SAMPLE_RATE)
            if begin + j < len(samples):
                samples[begin + j] += value
    pcm = [max(-32768, min(32767, int(v * 32767))) for v in samples]
    with wave.open(filename, 'wb') as w:
        w.setnchannels(1)
        w.setsampwidth(2)  # 16-bit PCM
        w.setframerate(SAMPLE_RATE)
        w.writeframes(struct.pack('<' + 'h' * len(pcm), *pcm))
    return len(pcm)

n = make_audio(EVENTS, 'midi_demo.wav')
print('Samples:', n)
print('Duration:', n / SAMPLE_RATE, 'seconds')
print('WAV bytes:', Path('midi_demo.wav').stat().st_size)
for _, _, note, velocity in EVENTS:
    print(note, velocity, round(midi_to_freq(note), 2), 'Hz')

# Listen directly inside Google Colab
display(Audio('midi_demo.wav'))

Samples: 77175
Duration: 1.75 seconds
WAV bytes: 154394
60 90 261.63 Hz
64 105 329.63 Hz
67 80 392.0 Hz


Cell 2 — Task 1B (Experimental Modifications): Pitch, Velocity & Duration
Runs the three required Task 1B variations into separate .wav files so the original midi_demo.wav remains intact for Tasks 2 and 3.

In [3]:
# 1. Change second note from 64 (E4) to 72 (C5)
events_pitch = [
    (0.00, 0.50, 60, 90),
    (0.50, 0.50, 72, 105),
    (1.00, 0.75, 67, 80),
]
make_audio(events_pitch, 'midi_mod1_pitch72.wav')
print("Mod 1 (Note 2 = 72) freq:", round(midi_to_freq(72), 2), "Hz")
display(Audio('midi_mod1_pitch72.wav'))

# 2. Change third velocity from 80 to 30
events_vel = [
    (0.00, 0.50, 60, 90),
    (0.50, 0.50, 64, 105),
    (1.00, 0.75, 67, 30),
]
make_audio(events_vel, 'midi_mod2_vel30.wav')
print("Mod 2 (Velocity 3 = 30) generated.")
display(Audio('midi_mod2_vel30.wav'))

# 3. Extend final note duration from 0.75 s to 1.50 s
events_dur = [
    (0.00, 0.50, 60, 90),
    (0.50, 0.50, 64, 105),
    (1.00, 1.50, 67, 80),
]
n_ext = make_audio(events_dur, 'midi_mod3_extended.wav')
print("Mod 3 (Extended 1.50s) -> Samples:", n_ext,
      "| Duration:", n_ext / SAMPLE_RATE, "s",
      "| PCM bytes:", n_ext * 2,
      "| WAV bytes:", Path('midi_mod3_extended.wav').stat().st_size)
display(Audio('midi_mod3_extended.wav'))

Mod 1 (Note 2 = 72) freq: 523.25 Hz


Mod 2 (Velocity 3 = 30) generated.


Mod 3 (Extended 1.50s) -> Samples: 110250 | Duration: 2.5 s | PCM bytes: 220500 | WAV bytes: 220544


Cell 3 — Task 2B (Part 1): Lossless zlib Compression & SHA-256 Verification
Tests zlib compression ratio and bit-exact round-trip recovery on highly repetitive vs. pseudo-random byte streams.

In [4]:
import zlib, hashlib

repetitive = (b'ABCD' * 10000)
# Deterministic pseudo-random-looking bytes (no extra packages needed)
less_repetitive = bytes((i * 73 + (i * i) % 251) % 256 for i in range(40000))

for name, data in [('repetitive', repetitive), ('less_repetitive', less_repetitive)]:
    packed = zlib.compress(data, level=9)
    recovered = zlib.decompress(packed)
    print(name)
    print('original:', len(data), 'compressed:', len(packed))
    print('ratio:', round(len(data) / len(packed), 2))
    print('exact recovery:', recovered == data)
    print('SHA-256:', hashlib.sha256(recovered).hexdigest())

repetitive
original: 40000 compressed: 66
ratio: 606.06
exact recovery: True
SHA-256: 0c78e5c9d34b7e2fba080cd1e507aa60d009f677e4a202a93bd901e1d76de6f1
less_repetitive
original: 40000 compressed: 40021
ratio: 1.0
exact recovery: True
SHA-256: 77a6d9a6f4c797c69c1ba2db218fe3115cab60ef5c4406a1fd35ef34c1c2c334


Cell 4 — Task 2B (Part 2): Lossless Compression of midi_demo.wav PCM Frames
Extracts the raw PCM sample payload (excluding the 44-byte WAV header) and verifies lossless zlib compression.

In [6]:
import wave, zlib

with wave.open('midi_demo.wav', 'rb') as w:
    frames = w.readframes(w.getnframes())

packed = zlib.compress(frames, 9)
print('PCM bytes:', len(frames))
print('zlib bytes:', len(packed))
print('Ratio:', round(len(frames) / len(packed), 2))
print('Lossless:', zlib.decompress(packed) == frames)

PCM bytes: 154350
zlib bytes: 139909
Ratio: 1.1
Lossless: True


Cell 5 — Task 3A: Lossy MP3 Encoding & Inspection via FFmpeg
Encodes midi_demo.wav to 64 kb/s and 128 kb/s MP3 files, inspects stream metadata via ffprobe, computes file-size ratios, and loads audio players.

In [7]:
!ffmpeg -y -i midi_demo.wav -codec:a libmp3lame -b:a 64k midi_64k.mp3
!ffmpeg -y -i midi_demo.wav -codec:a libmp3lame -b:a 128k midi_128k.mp3

print("--- ffprobe: 64k MP3 ---")
!ffprobe -v error -show_entries format=duration,size,bit_rate -of default=noprint_wrappers=1 midi_64k.mp3

print("\n--- ffprobe: 128k MP3 ---")
!ffprobe -v error -show_entries format=duration,size,bit_rate -of default=noprint_wrappers=1 midi_128k.mp3

from pathlib import Path
from IPython.display import Audio, display

wav_bytes = Path('midi_demo.wav').stat().st_size
mp3_64_bytes = Path('midi_64k.mp3').stat().st_size
mp3_128_bytes = Path('midi_128k.mp3').stat().st_size

print(f"\nWAV size:     {wav_bytes} B")
print(f"MP3 64k size: {mp3_64_bytes} B (Ratio: {wav_bytes / mp3_64_bytes:.2f}:1)")
print(f"MP3 128k size:{mp3_128_bytes} B (Ratio: {wav_bytes / mp3_128_bytes:.2f}:1)")

print("64 kb/s MP3:")
display(Audio('midi_64k.mp3'))
print("128 kb/s MP3:")
display(Audio('midi_128k.mp3'))

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

128 kb/s MP3:


Cell 6 — Task 4 (Part 1): Video Spatial & Temporal Redundancy (Small Motion)
Demonstrates intra-frame spatial compression vs. inter-frame XOR residual compression for a rectangle moving 2 pixels per frame.

Cell 6 — Task 4 (Part 1): Video Spatial & Temporal Redundancy (Small Motion)
Demonstrates intra-frame spatial compression vs. inter-frame XOR residual compression for a rectangle moving 2 pixels per frame.

In [8]:
import zlib

W, H = 160, 120

def frame(offset):
    pixels = bytearray([45] * (W * H))  # uniform background
    for y in range(35, 65):
        for x in range(20 + offset, 50 + offset):
            pixels[y * W + x] = 220   # moving bright rectangle
    return bytes(pixels)

frames = [frame(0), frame(2), frame(4), frame(6), frame(8)]
for i, current in enumerate(frames):
    full_size = len(zlib.compress(current, 9))
    if i == 0:
        print(i, 'compressed full frame:', full_size, '(reference frame)')
    else:
        previous = frames[i - 1]
        # XOR residual; a simple illustration, not a real video codec
        residual = bytes(a ^ b for a, b in zip(current, previous))
        residual_size = len(zlib.compress(residual, 9))
        print(i, 'full:', full_size, 'XOR residual:', residual_size)
        assert bytes(a ^ b for a, b in zip(previous, residual)) == current

0 compressed full frame: 63 (reference frame)
1 full: 63 XOR residual: 67
2 full: 63 XOR residual: 67
3 full: 63 XOR residual: 67
4 full: 63 XOR residual: 67


Cell 7 — Task 4 (Part 2): Video Redundancy Modification (Large 40-Pixel Jump)
Modifies the rectangle displacement to jump 40 pixels per frame (with horizontal bounds checking so pixels beyond W = 160 clip cleanly at the frame edge).

In [9]:
import zlib

W, H = 160, 120

def frame_clipped(offset):
    pixels = bytearray([45] * (W * H))
    for y in range(35, 65):
        for x in range(20 + offset, 50 + offset):
            if 0 <= x < W:  # prevent out-of-bounds wrap when x >= 160
                pixels[y * W + x] = 220
    return bytes(pixels)

frames_large_jump = [frame_clipped(i * 40) for i in range(5)]
for i, current in enumerate(frames_large_jump):
    full_size = len(zlib.compress(current, 9))
    if i == 0:
        print(i, 'compressed full frame:', full_size, '(reference frame)')
    else:
        previous = frames_large_jump[i - 1]
        residual = bytes(a ^ b for a, b in zip(current, previous))
        residual_size = len(zlib.compress(residual, 9))
        changed_pixels = sum(1 for b in residual if b != 0)
        print(f"{i} (offset {i*40}) -> full: {full_size} | XOR residual: {residual_size} | changed pixels: {changed_pixels}")
        assert bytes(a ^ b for a, b in zip(previous, residual)) == current

0 compressed full frame: 63 (reference frame)
1 (offset 40) -> full: 63 | XOR residual: 68 | changed pixels: 1800
2 (offset 80) -> full: 63 | XOR residual: 67 | changed pixels: 1800
3 (offset 120) -> full: 62 | XOR residual: 66 | changed pixels: 1500
4 (offset 160) -> full: 43 | XOR residual: 61 | changed pixels: 600
